# Set up individual pairwise comparisons

In [1]:
from itertools import combinations
from pathlib import Path
import pandas as pd
import glob
import nibabel as nib
from randomise_prep import setup_randomise_tfce

from survey_medley_code.config_loader import load_config

In [2]:
cfg = load_config()

In [4]:
mask_path = (
    cfg.output_root / 'assess_subject_bold_dropout/group_mask_intersection_30pct.nii.gz'
)
question_output_path = (
    cfg.output_root / 'within_subject_brain_behavior_by_questionnaire/within_subject_results'
)
outlier_assessment = question_output_path / 'outlier_assessment'

In [5]:
sub_id = []
bold_paths = []
questionnaire = []

questionnaire_names = ['grit', 'brief', 'future_time', 'impulsive_venture', 'upps']

for questionnaire_name in questionnaire_names:
    good_sub_file = (
        outlier_assessment
        / f'subjects_outlier_percent_lt_8_contrast_{questionnaire_name}.txt'
    )
    good_subs_list = good_sub_file.read_text().splitlines()
    for good_sub in good_subs_list:
        sub_no_s = good_sub.replace('s', '')
        # Glob for matching files
        bold_paths_loop = list(
            question_output_path.glob(
                f'{sub_no_s}/{questionnaire_name}_behavioral_measures_effect_size_sub_{sub_no_s}.nii.gz'
            )
        )
        if bold_paths_loop:
            sub_id.append(sub_no_s)
            questionnaire.append(questionnaire_name)
            bold_paths.append(bold_paths_loop[0])  # take the first match
        else:
            print(
                f'Output file missing for questionnaire/subject: {questionnaire_name}/{good_sub}'
            )

In [8]:
# Directory for outputs
base_out = Path(
    '/oak/stanford/groups/russpold/data/uh2/aim1/derivatives/'
    'survey_medley_results/within_subject_brain_behavior_by_questionnaire/'
    'all_paired_t_tests/paired_differences'
)

# -----------------------------------------------------
# BUILD A LOOKUP:  (sub, questionnaire) → file path
# -----------------------------------------------------

lookup = {}
for sid, qp, qname in zip(sub_id, bold_paths, questionnaire):
    lookup[(sid, qname)] = qp

# All questionnaire names present
questionnaire_names = sorted(set(questionnaire))

# -----------------------------------------------------
# LOOP OVER ALL QUESTIONNAIRE PAIRS
# -----------------------------------------------------
for qa, qb in combinations(questionnaire_names, 2):
    pairing_name = f'{qb}_minus_{qa}'
    outdir = base_out / pairing_name
    outdir.mkdir(parents=True, exist_ok=True)

    print(f'Processing pairing: {pairing_name}')

    # For every subject, check whether both images exist
    for sid in sub_id:
        keyA = (sid, qa)
        keyB = (sid, qb)

        if keyA not in lookup or keyB not in lookup:
            continue  # skip subjects missing data

        imgA = nib.load(str(lookup[keyA]))
        imgB = nib.load(str(lookup[keyB]))

        diff = imgB.get_fdata() - imgA.get_fdata()

        # Save output file
        out_path = outdir / f'{pairing_name}_{sid}.nii.gz'

        nib.Nifti1Image(diff, imgA.affine, imgA.header).to_filename(str(out_path))

        print(f'Saved {out_path}')

Processing pairing: future_time_minus_brief
Saved /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_differences/future_time_minus_brief/future_time_minus_brief_130.nii.gz
Saved /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_differences/future_time_minus_brief/future_time_minus_brief_172.nii.gz
Saved /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_differences/future_time_minus_brief/future_time_minus_brief_192.nii.gz
Saved /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_differences/future_time_minus_brief/future_time_minus_brief_234.nii.gz
Saved /oak/stanford/groups/russpold/data/uh2

In [11]:
base_dir = Path(
    '/oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire'
)
paired_diff_root = base_dir / 'all_paired_t_tests' / 'paired_differences'
randomise_root = base_dir / 'all_paired_t_tests'

for q1, q2 in combinations(questionnaire_names, 2):
    # --------------------------------------------
    # 1. Build the pairing name and collect files
    # --------------------------------------------
    pair_name = f'{q2}_minus_{q1}'
    diff_dir = paired_diff_root / pair_name

    bold_paths_loop = sorted(diff_dir.glob('*.nii.gz'))
    if not bold_paths_loop:
        print(f'⚠️ No paired difference images found for {pair_name}, skipping.')
        continue

    print(f'Found {len(bold_paths_loop)} paired images for {pair_name}')

    # --------------------------------------------
    # 2. Build output randomise directory
    # --------------------------------------------
    output_paired_ttest_loop = randomise_root / f'paired_test_{pair_name}'
    output_paired_ttest_loop.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------
    # 3. Run TFCE randomise job
    # --------------------------------------------
    script_path = setup_randomise_tfce(
        input_files=bold_paths_loop,
        group_mask=mask_path,  # already defined earlier
        output_directory=output_paired_ttest_loop,
        analysis_type='onesample_2sided',
        num_perm=5000,
        rename_output=True,
    )

    print(f'✓ Created randomise script for {pair_name}: {script_path}')

    module_lines = [
    "module load contribs\n",
    "module load poldrack\n",
    "module load fsl/6.0.7.10\n",
    "\n",
    ]
    with open(script_path, "r") as f:
        lines = f.readlines()
    new_lines = [lines[0]] + module_lines + lines[1:]
    with open(script_path, "w") as f:
        f.writelines(new_lines)

    print(f"✓ Added FSL module setup to {script_path}")
    

Found 90 paired images for future_time_minus_brief
Concatenating input files...
Created 4D file: /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_test_future_time_minus_brief/input_data4d.nii.gz
Note: rename_output=True will rename _corrp_ to _1minuspvalue_ in output filenames for t-test analysis
✓ Created randomise script for future_time_minus_brief: /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_test_future_time_minus_brief/randomise_call.sh
✓ Added FSL module setup to /oak/stanford/groups/russpold/data/uh2/aim1/derivatives/survey_medley_results/within_subject_brain_behavior_by_questionnaire/all_paired_t_tests/paired_test_future_time_minus_brief/randomise_call.sh
Found 92 paired images for grit_minus_brief
Concatenating input files...
Created 4D file: /oak/stanford/groups/russpold/